In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from datetime import datetime, time
from scipy.stats import ttest_1samp, ttest_rel

In [ ]:
# Read in the data
file_dir = './franklin rd/'
file_data = file_dir + '_data.csv'
file_metadata = file_dir + 'metadata.csv'

UTC_offset = -6  # UTC offset for local time conversion MST is -6

save_csv = False
plot=True
rolling=False

#______________________________________________________________________________

y_plot = ['Speed(miles/hour)']

df = pd.read_csv(file_data)
dfm = pd.read_csv(file_metadata)

# Convert the 'Date Time' column to datetime format
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%Y-%m-%dT%H:%M:%S%z',utc=True)

# Add columns for day of week and time of day
df['Day of Week'] = df['Date Time'].dt.dayofweek
df['Time of Day'] = df['Date Time'].dt.time

# Filter rows where 'CValue' > 80
df_filtered = df[df['CValue'] > 80]

# Calculate the average speed of the previous 5 and next 5 occurrences based on time of day and day of week
def calculate_moving_averages(group):
    group = group.sort_values(by='Date Time')
    group['Avg Speed (Prev 5)'] = group['Speed(miles/hour)'].rolling(window=5, min_periods=1).mean().shift(1)
    group['Avg Speed (Next 5)'] = group['Speed(miles/hour)'].shift(-1).rolling(window=5, min_periods=1).mean()
    return group

if rolling:
    df_filtered = df_filtered.groupby(['Segment ID', 'Day of Week', 'Time of Day'], group_keys=False).apply(calculate_moving_averages)
    y_plot.append(['Avg Speed (Prev 5)', 'Avg Speed (Next 5)'])

# Ensure Road, Direction, and Intersection are strings, replacing NaN with ''
dfm['Road'] = dfm['Road'].fillna('').astype(str)
dfm['Direction'] = dfm['Direction'].fillna('').astype(str)
dfm['Intersection'] = dfm['Intersection'].fillna('').astype(str)

# Concatenate 'Road', 'Direction', and 'Intersection' in dfm
dfm['Combined'] = dfm[['Road', 'Direction', 'Intersection']].agg(' '.join, axis=1)

# Perform a left join on 'Segment ID'
df_final = pd.merge(df_filtered, dfm[['Segment ID', 'Combined', 'Corridor']], on='Segment ID', how='left')
df_final["Date Time"] = df_final["Date Time"].dt.tz_convert(None)

if UTC_offset:
    df_final.loc[:, 'Date Time'] = df_final['Date Time'] + pd.Timedelta(hours=UTC_offset)

if save_csv:
    # Save the result to a new file
    df_final.to_csv(file_dir + 'processed_data.csv', index=False)

if plot:
    # Plot and save Speed, Avg Speed (Prev 5), and Avg Speed (Next 5) by Date Time for each Segment ID
    unique_segments = df_final['Segment ID'].unique()
    for segment_id in unique_segments:
        segment_data = df_final[df_final['Segment ID'] == segment_id].sort_values(by='Date Time')
        title = segment_data['Combined'].iloc[0] if not segment_data['Combined'].isna().all() else f'Segment ID: {segment_id}'
        title = segment_data['Corridor'].iloc[0] + ' - ' + title
        fig = px.line(
            segment_data,
            x='Date Time',
            y=y_plot,
            labels={"value": "Speed (miles/hour)", "variable": "Metric"},
            title=title
        )
        fig.write_html(f'{file_dir}{str(segment_data["Corridor"].iloc[0])}_segment_{segment_id}_speed_plot.html')

    print("Data processing and plotting complete. Plots saved.")

Data processing and plotting complete. Plots saved.


In [2]:
import pandas as pd
import plotly.express as px
from datetime import datetime



def process_and_plot_speed(
    file_dir,
    time_zone='America/Denver',
    save_csv=False,
    plot=True,
    rolling=False,
    bins=["6:20AM-9:00AM", "9:00AM-2:00PM", "2:00PM-7:00PM", "7:00PM-9:00PM"],
    day_groups=["Monday-Thursday", "Friday", "Saturday", "Sunday"]
):
    """
    Process and plot speed data with optional rolling averages and combined time/day grouping.

    Args:
        file_dir (str): Directory containing '__data.csv' and 'metadata.csv'.
        UTC_offset (int): Offset from UTC to local time (e.g., -6 for MST).
        save_csv (bool): Whether to save the processed CSV.
        plot (bool): Whether to generate and save scatter plots.
        rolling (bool): Whether to include rolling average columns.
        bins (list): List of time ranges (e.g., ["6:20AM-9:00AM", "9:00AM-2:00PM"]).
        day_groups (list): Day-of-week groups (e.g., ["Monday-Thursday", "Friday", "Saturday", "Sunday"]).
    """

    #---------------------------------------------------------------------------
    # Read in the data
    file_data = file_dir + 'data.csv'
    file_metadata = file_dir + 'metadata.csv'

    y_plot = ['Speed(miles/hour)']

    df = pd.read_csv(file_data)
    dfm = pd.read_csv(file_metadata)

    # Convert the 'Date Time' column to datetime format
    df['Date Time'] = pd.to_datetime(df['Date Time'], format='%Y-%m-%dT%H:%M:%S%z', utc=True)

    # Add columns for day of week and time of day
    df['Day of Week'] = df['Date Time'].dt.dayofweek  # Monday=0
    df['Time of Day'] = df['Date Time'].dt.time

    # Filter rows where 'CValue' > 80
    df_filtered = df[df['CValue'] > 80]

    #---------------------------------------------------------------------------
    # Optional rolling averages
    def calculate_moving_averages(group):
        group = group.sort_values(by='Date Time')
        group['Avg Speed (Prev 5)'] = group['Speed(miles/hour)'].rolling(window=5, min_periods=1).mean().shift(1)
        group['Avg Speed (Next 5)'] = group['Speed(miles/hour)'].shift(-1).rolling(window=5, min_periods=1).mean()
        return group

    if rolling:
        df_filtered = (
            df_filtered.groupby(['Segment ID', 'Day of Week'], group_keys=False)
            .apply(calculate_moving_averages)
        )
        y_plot.extend(['Avg Speed (Prev 5)', 'Avg Speed (Next 5)'])

    #---------------------------------------------------------------------------
    # Merge metadata
    dfm['Road'] = dfm['Road'].fillna('').astype(str)
    dfm['Direction'] = dfm['Direction'].fillna('').astype(str)
    dfm['Intersection'] = dfm['Intersection'].fillna('').astype(str)
    dfm['Combined'] = dfm[['Road', 'Direction', 'Intersection']].agg(' '.join, axis=1)

    df_final = pd.merge(df_filtered, dfm[['Segment ID', 'Combined', 'Corridor']], on='Segment ID', how='left')

    # Convert to local time zone
    if isinstance(time_zone, (int, float)):
        df_final["Date Time"] = df_final["Date Time"].dt.tz_convert(None)
        df_final.loc[:, 'Date Time'] = df_final['Date Time'] + pd.Timedelta(hours=time_zone)
    else:
        df_final["Date Time"] = df_final["Date Time"].dt.tz_convert(time_zone).dt.tz_localize(None)

    #---------------------------------------------------------------------------
    # Define helper functions for parsing and binning
    def parse_time(t_str):
        return datetime.strptime(t_str, "%I:%M%p").time()

    # Time bins
    time_bins = []
    bin_labels = []
    for b in bins:
        start_str, end_str = b.split('-')
        start_time, end_time = parse_time(start_str), parse_time(end_str)
        time_bins.append((start_time, end_time))
        bin_labels.append(b)

    # Day-of-week bins
    day_mapping = {
        "Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
        "Friday": 4, "Saturday": 5, "Sunday": 6
    }

    day_groups_parsed = []
    for g in day_groups:
        if '-' in g:
            start, end = g.split('-')
            start_idx, end_idx = day_mapping[start], day_mapping[end]
            if start_idx <= end_idx:
                day_range = list(range(start_idx, end_idx + 1))
            else:
                # handle wrap-around cases like "Friday-Monday"
                day_range = list(range(start_idx, 7)) + list(range(0, end_idx + 1))
        else:
            day_range = [day_mapping[g]]
        day_groups_parsed.append((g, day_range))

    # Assign time bin
    def assign_time_bin(row_time):
        for i, (start, end) in enumerate(time_bins):
            if start <= row_time <= end:
                return bin_labels[i]
        return None

    # Assign day bin
    def assign_day_bin(dow):
        for name, days in day_groups_parsed:
            if dow in days:
                return name
        return None

    df_final['Time Bin'] = df_final['Date Time'].dt.time.apply(assign_time_bin)
    df_final['Day Bin'] = df_final['Date Time'].dt.dayofweek.apply(assign_day_bin)

    # Combine both
    df_final['Group'] = df_final.apply(
        lambda r: f"{r['Day Bin']} {r['Time Bin']}" if pd.notna(r['Time Bin']) and pd.notna(r['Day Bin']) else None,
        axis=1
    )

    df_final = df_final.dropna(subset=['Group'])

    #---------------------------------------------------------------------------
    # Save processed data if needed
    if save_csv:
        df_final.to_csv(file_dir + 'processed_data.csv', index=False)

    #---------------------------------------------------------------------------
    # Plot scatter by Segment ID and Group
    if plot:
        unique_segments = df_final['Segment ID'].unique()
        for segment_id in unique_segments:
            segment_data = df_final[df_final['Segment ID'] == segment_id].sort_values(by='Date Time')
            title = segment_data['Combined'].iloc[0] if not segment_data['Combined'].isna().all() else f'Segment ID: {segment_id}'
            title = segment_data['Corridor'].iloc[0] + ' - ' + title

            fig = px.scatter(
                segment_data,
                x='Date Time',
                y='Speed(miles/hour)',
                color='Group',
                title=title,
                labels={'Speed(miles/hour)': 'Speed (miles/hour)'},
                hover_data=['CValue', 'Day of Week']
            )

            fig.update_layout(
                legend_title_text='Day/Time Group',
                legend=dict(itemsizing='trace', title_font=dict(size=12))
            )

            fig.write_html(f'{file_dir}{segment_id}_speed_scatter.html')

        print("Data processing and plotting complete. Plots saved.")

    return df_final


In [3]:
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

def process_and_plot_timebin_daily_summary(
    file_dir,
    time_zone='America/Denver',
    save_csv=False,
    rolling=False,
    bins=["6:20AM-9:00AM", "9:00AM-2:00PM", "2:00PM-7:00PM", "7:00PM-9:00PM"],
    day_groups=["Monday-Thursday", "Friday", "Saturday", "Sunday"]
):
    """
    Process speed data and plot daily mean ± 1 SD for each time-bin over time,
    combining day-groups and time-bins into a single interactive figure per segment.
    """
    
    # --------------------------
    # Load data
    file_data = file_dir + 'data.csv'
    file_metadata = file_dir + 'metadata.csv'
    df = pd.read_csv(file_data)
    dfm = pd.read_csv(file_metadata)

    df['Date Time'] = pd.to_datetime(df['Date Time'], format='%Y-%m-%dT%H:%M:%S%z', utc=True)
    df['Day of Week'] = df['Date Time'].dt.dayofweek
    df['Time of Day'] = df['Date Time'].dt.time
    df_filtered = df[df['CValue'] > 80]

    # --------------------------
    # Optional rolling averages
    def calculate_moving_averages(group):
        group = group.sort_values(by='Date Time')
        group['Avg Speed (Prev 5)'] = group['Speed(miles/hour)'].rolling(window=5, min_periods=1).mean().shift(1)
        group['Avg Speed (Next 5)'] = group['Speed(miles/hour)'].shift(-1).rolling(window=5, min_periods=1).mean()
        return group

    if rolling:
        df_filtered = df_filtered.groupby(['Segment ID', 'Day of Week'], group_keys=False).apply(calculate_moving_averages)

    # --------------------------
    # Merge metadata
    dfm[['Road','Direction','Intersection']] = dfm[['Road','Direction','Intersection']].fillna('').astype(str)
    dfm['Combined'] = dfm[['Road','Direction','Intersection']].agg(' '.join, axis=1)
    df_final = pd.merge(df_filtered, dfm[['Segment ID','Combined','Corridor']], on='Segment ID', how='left')

    # Convert to local time zone
    if isinstance(time_zone, (int, float)):
        df_final["Date Time"] = df_final["Date Time"].dt.tz_convert(None)
        df_final.loc[:, 'Date Time'] = df_final['Date Time'] + pd.Timedelta(hours=time_zone)
    else:
        df_final["Date Time"] = df_final["Date Time"].dt.tz_convert(time_zone).dt.tz_localize(None)

    # --------------------------
    # Time binning
    def parse_time(t_str):
        return datetime.strptime(t_str, "%I:%M%p").time()
    time_bins = [(parse_time(b.split('-')[0]), parse_time(b.split('-')[1])) for b in bins]
    bin_labels = bins

    def assign_time_bin(t):
        for i, (start, end) in enumerate(time_bins):
            if start <= t <= end:
                return bin_labels[i]
        return None
    df_final['Time Bin'] = df_final['Date Time'].dt.time.apply(assign_time_bin)

    # --------------------------
    # Day-of-week grouping
    day_mapping = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6}
    day_groups_parsed = []
    for g in day_groups:
        if '-' in g:
            start, end = g.split('-')
            start_idx, end_idx = day_mapping[start], day_mapping[end]
            if start_idx <= end_idx:
                days = list(range(start_idx,end_idx+1))
            else:
                days = list(range(start_idx,7)) + list(range(0,end_idx+1))
        else:
            days = [day_mapping[g]]
        day_groups_parsed.append((g, days))

    def assign_day_group(dow):
        for name, days in day_groups_parsed:
            if dow in days:
                return name
        return None
    df_final['Day Group'] = df_final['Date Time'].dt.dayofweek.apply(assign_day_group)

    # Drop rows without valid bins or day groups
    df_final = df_final.dropna(subset=['Time Bin','Day Group'])
    df_final['Group'] = df_final['Day Group'] + " " + df_final['Time Bin']

    # --------------------------
    # Aggregate by date and group
    df_final['Date'] = df_final['Date Time'].dt.date
    daily_stats = (
        df_final.groupby(['Segment ID','Group','Date'])['Speed(miles/hour)']
        .agg(['mean','std']).reset_index()
    )
    daily_stats.rename(columns={'mean':'Mean','std':'Std'}, inplace=True)
    daily_stats['Upper'] = daily_stats['Mean'] + daily_stats['Std']
    daily_stats['Lower'] = daily_stats['Mean'] - daily_stats['Std']

    if save_csv:
        daily_stats.to_csv(file_dir+'daily_summary_timebin_combined.csv', index=False)

    # --------------------------
    # Plot per segment
    for segment_id, seg in daily_stats.groupby('Segment ID'):
        segment_name = df_final[df_final['Segment ID']==segment_id]['Combined'].iloc[0]
        corridor = df_final[df_final['Segment ID']==segment_id]['Corridor'].iloc[0]

        fig = go.Figure()
        for grp, gseg in seg.groupby('Group'):
            fig.add_trace(go.Scatter(
                x=gseg['Date'],
                y=gseg['Mean'],
                mode='lines+markers',
                name=grp,
                line=dict(width=2),
                error_y=dict(type='data', array=gseg['Std'], visible=True)
            ))

        fig.update_layout(
            title=f"{corridor} - {segment_name} Daily Mean ± 1 SD by Time Bin & Day Group",
            xaxis_title="Date",
            yaxis_title="Speed (miles/hour)",
            legend_title_text="Day Group + Time Bin"
        )

        fig.write_html(f"{file_dir}{segment_id}_daily_summary.html")

    print("Combined daily summary plot saved.")
    return daily_stats



In [4]:
process_and_plot_timebin_daily_summary(
    file_dir='./eagle rd/',  time_zone='America/Denver', save_csv=False, rolling=False,
    bins=["6:30AM-8:00AM", "8:00AM-10:00AM","10:00AM-2:00PM", "2:00PM-4:00PM", "4:00PM-7:00PM", "7:00PM-9:00PM"],
    day_groups=["Monday-Thursday","Friday", "Saturday", "Sunday"])

Combined daily summary plot saved.


,Segment ID,Group,Date,Mean,Std,Upper,Lower
0,117503882,Friday 10:00AM-2:00PM,2023-04-07,25.7500,3.838402,29.588402,21.911598
1,117503882,Friday 10:00AM-2:00PM,2023-04-14,26.0625,2.322893,28.385393,23.739607
2,117503882,Friday 10:00AM-2:00PM,2023-04-21,25.6875,2.522400,28.209900,23.165100
3,117503882,Friday 10:00AM-2:00PM,2023-04-28,25.6875,2.548692,28.236192,23.138808
4,117503882,Friday 10:00AM-2:00PM,2023-05-05,25.1250,2.093641,27.218641,23.031359
...,...,...,...,...,...,...,...
273568,1187654609,Sunday 8:00AM-10:00AM,2025-08-03,42.0000,2.725541,44.725541,39.274459
273569,1187654609,Sunday 8:00AM-10:00AM,2025-08-10,41.3750,4.596194,45.971194,36.778806
273570,1187654609,Sunday 8:00AM-10:00AM,2025-08-17,41.2500,3.615443,44.865443,37.634557
273571,1187654609,Sunday 8:00AM-10:00AM,2025-08-24,41.5000,4.440077,45.940077,37.059923


In [4]:
# Read in the data
file_dir = './franklin rd/'
file_data = file_dir + 'data.csv'
file_metadata = file_dir + 'metadata.csv'

time_zone = 'America/Denver'  # Time zone for local time conversion
#time_zone = -6

save_csv = False
plot=True
rolling=False
plot_daily = True

bins = [
    "6:00AM-7:00AM",
    "7:00AM-8:00AM",
    "8:00AM-9:00AM",
    "9:00AM-10:00AM",
    "10:00AM-11:00AM",
    "11:00AM-12:00PM",
    "12:00PM-1:00PM",
    "1:00PM-2:00PM",
    "2:00PM-3:00PM",
    "3:00PM-4:00PM",
    "4:00PM-5:00PM",
    "5:00PM-6:00PM",
    "6:00PM-7:00PM",
    "7:00PM-8:00PM",
    "8:00PM-9:00PM",
    "9:00PM-10:00PM"
]
#______________________________________________________________________________

process_and_plot_speed(
    file_dir=file_dir,
    time_zone=time_zone,
    save_csv=save_csv,
    plot=plot,
    rolling=rolling,
    bins=bins,
    day_groups=["Monday-Thursday","Friday", "Saturday", "Sunday"]
)

if plot_daily:
    process_and_plot_timebin_daily_summary(
        file_dir=file_dir,
        time_zone=time_zone,
        save_csv=save_csv,
        rolling=rolling,
        bins=bins,
        day_groups=["Monday-Thursday","Friday", "Saturday", "Sunday"]
    )



Data processing and plotting complete. Plots saved.
Combined daily summary plot saved.


In [ ]:


process_and_plot_speed(
    file_dir,
    time_zone='America/Denver',
    save_csv=False,
    plot=False,
    rolling=False,
    bins=["6:20AM-9:00AM", "9:00AM-2:00PM", "2:00PM-7:00PM", "7:00PM-9:00PM"],
    day_groups=["Monday-Thursday", "Friday", "Saturday", "Sunday"]
)

df_sum = df_final.groupby(['Date Time', 'Corridor'])['Travel Time(Minutes)'].agg(['count', 'sum']).reset_index()
df_sum = pd.merge(df_sum, df_sum.groupby('Corridor')['count'].max(), on='Corridor', suffixes=('', '_max'))
df_sum = df_sum.loc[(df_sum['count'] == df_sum['count_max']), ['Date Time', 'Corridor', 'sum']]
df_sum.rename(columns={'sum':'Travel Time(Minutes)', 'Corridor':'Segment ID'}, inplace=True)
df_sum.loc[:, 'Day of Week'] = df_sum['Date Time'].dt.dayofweek

In [5]:
def map_day_group(dow: int) -> str:
    if dow <= 3:   # Mon (0) – Thu (3)
        return "Mon–Thu"
    elif dow == 4:
        return "Fri"
    elif dow == 5:
        return "Sat"
    else:
        return "Sun"

def assign_time_chunks(df, time_chunks):
    """
    Assigns each row to a (Day Group, Time Chunk).
    time_chunks is a list like ['6:30AM-9:00AM', '9:00AM-12:30PM', ...]
    """
    # Parse time chunks into (label, start, end)
    parsed_chunks = []
    for chunk in time_chunks:
        start_str, end_str = chunk.split('-')
        start = datetime.strptime(start_str.strip(), "%I:%M%p").time()
        end   = datetime.strptime(end_str.strip(), "%I:%M%p").time()
        parsed_chunks.append((chunk, start, end))

    def map_chunk(t):
        for label, start, end in parsed_chunks:
            if start <= end:
                if start <= t < end:
                    return label
            else:  # overnight (e.g. 9PM-6AM)
                if t >= start or t < end:
                    return label
        return "Unassigned"

    df = df.copy()
    df["Day Group"] = df["Date Time"].dt.dayofweek.map(map_day_group)
    df["Time Chunk"] = df["Date Time"].dt.time.apply(map_chunk)
    df["Group Label"] = df["Day Group"] + ", " + df["Time Chunk"]
    return df

def analyze_travel_time(df_final, time_chunks, before_period, after_period):
    """
    Compare before/after periods for travel time differences.
    Aggregates by 15-min time-of-day bins within each DayGroup x TimeChunk.
    Performs a paired t-test across bins.
    """
    # Parse periods
    before_start, before_end = before_period.split('-')
    after_start, after_end   = after_period.split('-')

    before_start = pd.to_datetime(before_start, format="%Y%m%d")
    before_end   = pd.to_datetime(before_end,   format="%Y%m%d")
    after_start  = pd.to_datetime(after_start,  format="%Y%m%d")
    after_end    = pd.to_datetime(after_end,   format="%Y%m%d")

    # Assign chunks and day groups
    df = assign_time_chunks(df_final, time_chunks)

    # Split periods
    df_before = df[(df["Date Time"] >= before_start) & (df["Date Time"] <= before_end)].copy()
    df_after  = df[(df["Date Time"] >= after_start) & (df["Date Time"] <= after_end)].copy()

    # Create 15-min time-of-day bins
    df_before["TOD Bin"] = df_before["Date Time"].dt.floor("15min").dt.time
    df_after["TOD Bin"]  = df_after["Date Time"].dt.floor("15min").dt.time

    results = []

    # Loop over segment × group label
    for (segment, group_label), group_before in df_before.groupby(["Segment ID", "Group Label"]):
        group_after = df_after[(df_after["Segment ID"] == segment) & (df_after["Group Label"] == group_label)]
        if group_after.empty:
            continue

        # Aggregate by TOD bin (averaging across all days)
        before_means = group_before.groupby("TOD Bin")["Travel Time(Minutes)"].mean()
        after_means  = group_after.groupby("TOD Bin")["Travel Time(Minutes)"].mean()

        # Align bins
        aligned = pd.DataFrame({"before": before_means, "after": after_means}).dropna()
        if aligned.empty:
            continue

        diffs = aligned["after"] - aligned["before"]

        # Paired t-test
        t_stat, p_val = ttest_rel(aligned["after"], aligned["before"])

        results.append({
            "Segment ID": segment,
            "Group Label": group_label,
            "Avg Difference (min)": diffs.mean(),
            "T-stat": t_stat,
            "P-value": p_val,
            "N bins": len(diffs)
        })

    return pd.DataFrame(results)

In [6]:
# Read in the data
file_dir = './franklin rd/'
file_data = file_dir + 'data.csv'
file_metadata = file_dir + 'metadata.csv'

time_zone = 'America/Denver'  # Time zone for local time conversion

periods=["20251022-20251023","20251117-20251118"]

save_csv = False
plot=False
rolling=False

df_final = process_and_plot_speed(
    file_dir,
    time_zone=time_zone,
    save_csv=save_csv,
    plot=plot,
    rolling=rolling,
    bins=["6:20AM-9:00AM", "9:00AM-2:00PM", "2:00PM-7:00PM", "7:00PM-9:00PM"],
    day_groups=["Monday-Thursday", "Friday", "Saturday", "Sunday"]
)

df_sum = df_final.groupby(['Date Time', 'Corridor'])['Travel Time(Minutes)'].agg(['count', 'sum']).reset_index()
df_sum = pd.merge(df_sum, df_sum.groupby('Corridor')['count'].max(), on='Corridor', suffixes=('', '_max'))
df_sum = df_sum.loc[(df_sum['count'] == df_sum['count_max']), ['Date Time', 'Corridor', 'sum']]
df_sum.rename(columns={'sum':'Travel Time(Minutes)', 'Corridor':'Segment ID'}, inplace=True)
df_sum.loc[:, 'Day of Week'] = df_sum['Date Time'].dt.dayofweek

before_period = periods[0]
after_period = periods[1]

days = [0,1,2,3,4,5,6]  # integers (0=Mon,...,6=Sun)

time_chunks = [
    "6:00AM-7:00AM",
    "7:00AM-8:00AM",
    "8:00AM-9:00AM",
    "9:00AM-10:00AM",
    "10:00AM-11:00AM",
    "11:00AM-12:00PM",
    "12:00PM-1:00PM",
    "1:00PM-2:00PM",
    "2:00PM-3:00PM",
    "3:00PM-4:00PM",
    "4:00PM-5:00PM",
    "5:00PM-6:00PM",
    "6:00PM-7:00PM",
    "7:00PM-8:00PM",
    "8:00PM-9:00PM",
    "9:00PM-10:00PM"
]


summary = analyze_travel_time(
    df_sum[df_sum["Day of Week"].isin(days)],
    time_chunks,
    before_period=before_period,
    after_period=after_period
)

print(summary)


c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

divide by zero encountered in divide

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

invalid value encountered in scalar multiply

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

divide by zero encountered in divide

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

invalid value encountered in scalar multiply

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

divide by zero encountered in divide

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

invalid value encountered in scalar multiply

c:\Users\r

      Segment ID               Group Label  Avg Difference (min)    T-stat  \
0    Aviation_NB  Mon–Thu, 10:00AM-11:00AM              0.103333  0.800150   
1    Aviation_NB  Mon–Thu, 11:00AM-12:00PM             -0.172500 -1.010815   
2    Aviation_NB   Mon–Thu, 12:00PM-1:00PM              0.127500  1.488245   
3    Aviation_NB    Mon–Thu, 1:00PM-2:00PM             -0.180000 -6.114296   
4    Aviation_NB    Mon–Thu, 2:00PM-3:00PM             -0.112500 -0.821082   
..           ...                       ...                   ...       ...   
143           WB    Mon–Thu, 7:00PM-8:00PM             -0.820000 -1.490786   
144           WB    Mon–Thu, 8:00AM-9:00AM              0.175000  1.114062   
145           WB    Mon–Thu, 8:00PM-9:00PM             -0.020000 -0.105409   
146           WB   Mon–Thu, 9:00AM-10:00AM              0.130000  1.337296   
147           WB   Mon–Thu, 9:00PM-10:00PM             -0.590000       NaN   

      P-value  N bins  
0    0.507564       3  
1    0.386554  

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

divide by zero encountered in divide

c:\Users\rhansen\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:1214: RuntimeWarning:

invalid value encountered in scalar multiply



In [8]:
summary

,Segment ID,Group Label,Avg Difference (min),T-stat,P-value,N bins
0,Aviation_NB,"Mon–Thu, 10:00AM-11:00AM",0.103333,0.800150,0.507564,3
1,Aviation_NB,"Mon–Thu, 11:00AM-12:00PM",-0.172500,-1.010815,0.386554,4
2,Aviation_NB,"Mon–Thu, 12:00PM-1:00PM",0.127500,1.488245,0.233424,4
3,Aviation_NB,"Mon–Thu, 1:00PM-2:00PM",-0.180000,-6.114296,0.008793,4
4,Aviation_NB,"Mon–Thu, 2:00PM-3:00PM",-0.112500,-0.821082,0.471769,4
...,...,...,...,...,...,...
143,WB,"Mon–Thu, 7:00PM-8:00PM",-0.820000,-1.490786,0.232807,4
144,WB,"Mon–Thu, 8:00AM-9:00AM",0.175000,1.114062,0.346473,4
145,WB,"Mon–Thu, 8:00PM-9:00PM",-0.020000,-0.105409,0.922704,4
146,WB,"Mon–Thu, 9:00AM-10:00AM",0.130000,1.337296,0.273479,4
